# Relatório de Dados - Proposta v0

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcelo7bastos/relatorio_dados_damei/blob/main/notebooks/000_proposta_v0.ipynb)

Notebook consolidado para leitura das bases atuais, cálculo de indicadores UF x Brasil e geração de relatório estadual em Word. Esta versão usa `config/ufs.csv` como dicionário oficial de UFs e aceita UF por sigla ou código IBGE.


## 1. Importar bibliotecas

Usamos bibliotecas padrão do Python para caminhos, datas e ambiente, além de `pandas` para ler as planilhas Excel.

In [ ]:
from datetime import datetime
from pathlib import Path
import os

import pandas as pd


## 2. Configurar origem dos dados e pasta de saída

O GitHub guarda código, documentação e template. O Google Drive é o repositório operacional dos dados e dos relatórios gerados.

Existem apenas dois modos:

- `local`: roda no VS Code e usa uma cópia local/mock em `dados_brutos/dado_atual`.
- `google_drive`: roda no Google Colab, monta o Google Drive e lê os dados oficiais do Drive.

In [ ]:
# O modo é escolhido automaticamente:
# - Google Colab: "google_drive"
# - VS Code/local: "local"
MODO_DADOS = "google_drive" if "COLAB_RELEASE_TAG" in os.environ else "local"

# Defina uma UF por sigla (ex.: "MG") ou código IBGE (ex.: "31").
# Use "ALL" para gerar relatórios para todas as UFs.
UF_INTERESSE = "PA"

# Quando True, ignora UF_INTERESSE e gera relatórios para todas as UFs do dicionário oficial.
GERAR_TODAS_UFS = False

# Ranking é opcional nesta versão; o relatório base prioriza tabelas, fontes e lacunas.
INCLUIR_RANKING = True

# Caminhos no Google Drive para execução no Colab.
# ID da pasta compartilhada raiz (DAMEI_Relatorio_Dados).
GOOGLE_DRIVE_SHARED_FOLDER_ID = "1vjphYN2JHf-OsvJdJ97NxxI1CdL_tXid"

# ID da pasta compartilhada que contém os arquivos Excel.
GOOGLE_DRIVE_EXCEL_FOLDER_ID = "1xGcxCFmkeRrETVpePgzzdvleaWKWr0IO"

# Fallback para estrutura antiga no seu MyDrive (caso o atalho não exista).
GOOGLE_DRIVE_RAW_DIR = Path(r"/content/drive/MyDrive/DAMEI_Relatorio_Dados atalho/dados_brutos/dado_atual")

# Saída operacional definida no PRD para execução em Google Drive.
GOOGLE_DRIVE_OUTPUT_DIR = Path(r"/content/drive/MyDrive/MDA/dado_atual")

# Subpastas usadas no repositório local.
RAW_SUBDIR = Path("dados_brutos") / "dado_atual"
OUTPUT_SUBDIR = Path("relatorios_gerados")
UF_TABLE_SUBPATH = Path("config") / "ufs.csv"

# Data de execução usada no nome dos arquivos gerados.
RUN_TIMESTAMP = datetime.now()
RUN_YYYYMM = RUN_TIMESTAMP.strftime("%Y%m")
RUN_DATETIME = RUN_TIMESTAMP.strftime("%Y%m%d%H%M%S")


## 3. Resolver caminhos

Esta célula transforma a configuração acima em caminhos finais de leitura e gravação.

No modo `local`, o notebook usa as pastas do próprio repositório. No modo `google_drive`, o notebook exige Google Colab, monta o Drive e usa os caminhos `/content/drive/...`.

In [ ]:
def esta_no_colab() -> bool:
    """Retorna True quando o notebook está rodando no Google Colab."""
    return "COLAB_RELEASE_TAG" in os.environ


def encontrar_raiz_projeto(inicio: Path | None = None) -> Path:
    """Procura a raiz do repositório a partir da pasta atual."""
    inicio = (inicio or Path.cwd()).resolve()

    for candidato in [inicio, *inicio.parents]:
        tem_notebooks = (candidato / "notebooks").exists()
        tem_requirements = (candidato / "requirements.txt").exists()
        tem_config_ufs = (candidato / UF_TABLE_SUBPATH).exists()
        if tem_notebooks and (tem_requirements or tem_config_ufs):
            return candidato

    for candidato in [inicio, *inicio.parents]:
        if (candidato / RAW_SUBDIR).exists():
            return candidato

    return inicio


CODE_PROJECT_ROOT = encontrar_raiz_projeto()

if MODO_DADOS == "local":
    PROJECT_ROOT = CODE_PROJECT_ROOT
    DATA_PROJECT_ROOT = PROJECT_ROOT

    raw_candidatos = [
        PROJECT_ROOT / RAW_SUBDIR,
        PROJECT_ROOT / "dados_brutos" / "dados_atual",
        PROJECT_ROOT / "dado_atual",
        PROJECT_ROOT,
    ]

    RAW_DIR = None
    for candidato in raw_candidatos:
        if candidato.exists() and candidato.is_dir() and list(candidato.glob("*.xlsx")):
            RAW_DIR = candidato
            break

    if RAW_DIR is None:
        for candidato in raw_candidatos:
            if candidato.exists() and candidato.is_dir():
                RAW_DIR = candidato
                break

    if RAW_DIR is None:
        RAW_DIR = PROJECT_ROOT / RAW_SUBDIR

    OUTPUT_DIR = PROJECT_ROOT / OUTPUT_SUBDIR / RUN_YYYYMM

elif MODO_DADOS == "google_drive":
    if not esta_no_colab():
        raise RuntimeError(
            'MODO_DADOS = "google_drive" só deve ser usado no Google Colab. '
            'No VS Code local, use MODO_DADOS = "local".'
        )

    from google.colab import drive

    drive.mount("/content/drive")

    def resolver_raiz_drive() -> Path:
        """Resolve a raiz DAMEI_Relatorio_Dados em cenários de pasta compartilhada."""
        nome_raiz = "DAMEI_Relatorio_Dados"
        candidatos = [
            Path("/content/drive/MyDrive/.shortcut-targets-by-id") / GOOGLE_DRIVE_SHARED_FOLDER_ID,
            Path("/content/drive/MyDrive") / nome_raiz,
            Path("/content/drive/MyDrive") / "MDA" / nome_raiz,
            Path("/content/drive/Shareddrives") / nome_raiz,
        ]

        for candidato in candidatos:
            if candidato.exists():
                return candidato

        raise FileNotFoundError(
            "Não foi possível localizar a pasta DAMEI_Relatorio_Dados no Google Drive. "
            "Adicione o atalho da pasta compartilhada ao Meu Drive ou ajuste GOOGLE_DRIVE_RAW_DIR."
        )

    excel_shared_dir = Path("/content/drive/MyDrive/.shortcut-targets-by-id") / GOOGLE_DRIVE_EXCEL_FOLDER_ID

    if excel_shared_dir.exists():
        RAW_DIR = excel_shared_dir
        DATA_PROJECT_ROOT = excel_shared_dir.parent
    else:
        DATA_PROJECT_ROOT = resolver_raiz_drive()
        raw_candidatos = [
            GOOGLE_DRIVE_RAW_DIR,
            DATA_PROJECT_ROOT / RAW_SUBDIR,
            DATA_PROJECT_ROOT / "dados_brutos" / "dados_atual",
            DATA_PROJECT_ROOT / "dado_atual",
            DATA_PROJECT_ROOT / "dados_atual",
            DATA_PROJECT_ROOT,
        ]

        RAW_DIR = None
        for candidato in raw_candidatos:
            if candidato.exists() and candidato.is_dir() and list(candidato.glob("*.xlsx")):
                RAW_DIR = candidato
                break

        if RAW_DIR is None:
            for arquivo in DATA_PROJECT_ROOT.rglob("*.xlsx"):
                RAW_DIR = arquivo.parent
                break

        if RAW_DIR is None:
            RAW_DIR = DATA_PROJECT_ROOT / RAW_SUBDIR

    OUTPUT_DIR = GOOGLE_DRIVE_OUTPUT_DIR
else:
    raise ValueError('MODO_DADOS deve ser "local" ou "google_drive".')

UF_TABLE_PATH = CODE_PROJECT_ROOT / UF_TABLE_SUBPATH
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Modo de dados:     {MODO_DADOS}")
print(f"UF de interesse:   {UF_INTERESSE}")
print(f"Raiz do código:    {CODE_PROJECT_ROOT}")
print(f"Projeto de dados:  {DATA_PROJECT_ROOT}")
print(f"Pasta de dados:    {RAW_DIR}")
print(f"Pasta de saída:    {OUTPUT_DIR}")
print(f"Dicionário de UFs: {UF_TABLE_PATH}")


## 4. Validar pastas de dados e saída

Antes de ler qualquer arquivo, validamos se a pasta de dados existe. A pasta de saída é criada automaticamente quando ainda não existir. No modo `google_drive`, ela usa o caminho configurado em `GOOGLE_DRIVE_OUTPUT_DIR`.

In [ ]:
if not RAW_DIR.exists():
    # Reavalia automaticamente o diretório de entrada quando o caminho padrão não existe.
    candidatos = [
        DATA_PROJECT_ROOT,
        DATA_PROJECT_ROOT / "dados_brutos",
        DATA_PROJECT_ROOT / "dados_brutos" / "dado_atual",
        DATA_PROJECT_ROOT / "dados_brutos" / "dados_atual",
    ]

    novo_raw = None
    for cand in candidatos:
        if cand.exists() and cand.is_dir() and list(cand.glob("*.xlsx")):
            novo_raw = cand
            break

    if novo_raw is None:
        for arq in DATA_PROJECT_ROOT.rglob("*.xlsx"):
            novo_raw = arq.parent
            break

    if novo_raw is not None:
        RAW_DIR = novo_raw
        print(f"RAW_DIR ajustado automaticamente para: {RAW_DIR}")
    else:
        raise FileNotFoundError(
            "A pasta de dados não existe e não foi possível localizar arquivos .xlsx "
            f"em {DATA_PROJECT_ROOT}."
        )

print("Pasta de dados encontrada.")
print("Pasta de saída pronta.")


## 5. Listar arquivos Excel

Aqui verificamos quais arquivos `.xlsx` estão disponíveis na pasta de dados configurada.

In [ ]:
excel_files = sorted(RAW_DIR.glob("*.xlsx"))

if not excel_files:
    raise FileNotFoundError(f"Nenhum arquivo .xlsx encontrado em {RAW_DIR}")

print(f"Foram encontrados {len(excel_files)} arquivos Excel:")
for arquivo in excel_files:
    tamanho_mb = arquivo.stat().st_size / (1024 * 1024)
    print(f"- {arquivo.name} ({tamanho_mb:.2f} MB)")


## 6. Identificar arquivos por política

Nesta etapa fazemos um primeiro mapeamento automático entre nomes de arquivos e políticas públicas. Essa lógica ainda é simples, mas já ajuda a validar se temos todas as bases esperadas.

In [ ]:
arquivos_encontrados = {
    "caf_dap": None,
    "ater": None,
    "pronaf": None,
    "mais_alimentos": None,
    "pncf": None,
    "garantia_safra": None,
    "pnra": None,
}

for arquivo in excel_files:
    nome = arquivo.name.lower()

    if "caf" in nome:
        arquivos_encontrados["caf_dap"] = arquivo
    elif "ater" in nome:
        arquivos_encontrados["ater"] = arquivo
    elif "pronaf" in nome:
        arquivos_encontrados["pronaf"] = arquivo
    elif "mais_alimentos" in nome or "mais-alimentos" in nome:
        arquivos_encontrados["mais_alimentos"] = arquivo
    elif "pncf" in nome:
        arquivos_encontrados["pncf"] = arquivo
    elif "garantia-safra" in nome or "garantia_safra" in nome:
        arquivos_encontrados["garantia_safra"] = arquivo
    elif "pnra" in nome:
        arquivos_encontrados["pnra"] = arquivo

resumo_arquivos = pd.DataFrame(
    [
        {
            "politica": politica,
            "arquivo": arquivo.name if arquivo else "NÃO ENCONTRADO",
            "encontrado": arquivo is not None,
        }
        for politica, arquivo in arquivos_encontrados.items()
    ]
)

resumo_arquivos


## 7. Inspecionar abas e colunas

Agora abrimos cada arquivo com `pandas.ExcelFile`, listamos as abas e coletamos os nomes das colunas. Este teste ainda não transforma os dados; ele apenas confirma que os arquivos são legíveis.

In [ ]:
linhas = []

for politica, arquivo in arquivos_encontrados.items():
    if arquivo is None:
        linhas.append(
            {
                "politica": politica,
                "arquivo": "NÃO ENCONTRADO",
                "aba": None,
                "qtd_colunas": None,
                "colunas": None,
            }
        )
        continue

    xls = pd.ExcelFile(arquivo)
    for aba in xls.sheet_names:
        colunas = pd.read_excel(arquivo, sheet_name=aba, nrows=0).columns.astype(str).tolist()
        linhas.append(
            {
                "politica": politica,
                "arquivo": arquivo.name,
                "aba": aba,
                "qtd_colunas": len(colunas),
                "colunas": ", ".join(colunas[:12]) + (" ..." if len(colunas) > 12 else ""),
            }
        )

df_estrutura = pd.DataFrame(linhas)
df_estrutura


## 8. Testar leitura das abas principais

Por fim, carregamos as abas principais de cada política e mostramos o tamanho de cada tabela. Se esta célula rodar sem erro, o teste de leitura está aprovado.

In [ ]:
abas_principais = {
    "caf_dap": "DADOS",
    "ater": "DADOS",
    "pronaf": "Dados",
    "mais_alimentos": "Dados",
    "pncf": "DADOS",
    "garantia_safra": "DADOS",
    "pnra": "DADOS",
}

tabelas = {}
resumo_leitura = []

for politica, aba in abas_principais.items():
    arquivo = arquivos_encontrados.get(politica)

    if arquivo is None:
        resumo_leitura.append(
            {
                "politica": politica,
                "status": "arquivo não encontrado",
                "linhas": None,
                "colunas": None,
            }
        )
        continue

    df = pd.read_excel(arquivo, sheet_name=aba)
    tabelas[politica] = df
    resumo_leitura.append(
        {
            "politica": politica,
            "status": "ok",
            "linhas": df.shape[0],
            "colunas": df.shape[1],
        }
    )

pd.DataFrame(resumo_leitura)


## 9. Incorporação da proposta Wesley

A partir daqui incorporamos os avanços do notebook `proposta_wesley.ipynb`: uso dos totalizadores de CAF e PRONAF, comparação UF x Brasil e geração de um primeiro relatório Word de teste. A implementação abaixo usa os caminhos já configurados neste notebook, sem caminhos fixos e sem download automático.

## 10. Funções auxiliares

Estas funções padronizam validação, percentuais e formatação brasileira de números antes da geração do relatório.

In [ ]:
try:
    from docx import Document
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])
    from docx import Document
    from docx.enum.text import WD_ALIGN_PARAGRAPH


UF_ALIASES_LOTE = {"ALL", "TODAS", "*"}


def validar_colunas(df: pd.DataFrame, colunas: list[str], contexto: str) -> None:
    faltantes = [col for col in colunas if col not in df.columns]
    if faltantes:
        raise KeyError(f"Colunas ausentes em {contexto}: {faltantes}")


def carregar_tabela_ufs(caminho: Path) -> pd.DataFrame:
    if not caminho.exists():
        raise FileNotFoundError(
            f"Dicionário de UFs não encontrado: {caminho}. "
            "Verifique se config/ufs.csv existe no repositório clonado."
        )

    df = pd.read_csv(caminho, dtype={"sigla": "string", "codigo_ibge": "string", "nome_uf": "string"})
    validar_colunas(df, ["sigla", "codigo_ibge", "nome_uf"], "config/ufs.csv")

    df = df.copy()
    df["sigla"] = df["sigla"].astype(str).str.strip().str.upper()
    df["codigo_ibge"] = df["codigo_ibge"].astype(str).str.strip().str.zfill(2)
    df["nome_uf"] = df["nome_uf"].astype(str).str.strip()

    duplicadas = df[df.duplicated(["sigla"], keep=False) | df.duplicated(["codigo_ibge"], keep=False)]
    if not duplicadas.empty:
        raise ValueError("Dicionário de UFs contém siglas ou códigos IBGE duplicados.")

    if len(df) != 27:
        raise ValueError(f"Dicionário de UFs deveria conter 27 linhas, mas contém {len(df)}.")

    return df


df_ufs = carregar_tabela_ufs(UF_TABLE_PATH)
UF_VALIDAS = set(df_ufs["sigla"])
UF_NOMES = dict(zip(df_ufs["sigla"], df_ufs["nome_uf"]))
UF_CODIGOS_PARA_SIGLA = dict(zip(df_ufs["codigo_ibge"], df_ufs["sigla"]))


def normalizar_uf(uf: str) -> str:
    texto = str(uf).strip().upper()
    if texto in UF_ALIASES_LOTE:
        return texto

    codigo = texto.zfill(2) if texto.isdigit() else texto
    if codigo in UF_CODIGOS_PARA_SIGLA:
        return UF_CODIGOS_PARA_SIGLA[codigo]

    if texto in UF_VALIDAS:
        return texto

    raise ValueError(
        f"UF inválida: {uf}. Informe uma sigla válida, um código IBGE de UF ou ALL para lote."
    )


def filtrar_ufs_validas(df: pd.DataFrame, coluna_uf: str = "uf", contexto: str = "dados") -> pd.DataFrame:
    validar_colunas(df, [coluna_uf], contexto)
    serie_uf = df[coluna_uf].astype(str).str.strip().str.upper()
    invalidas = sorted(set(serie_uf.dropna()) - UF_VALIDAS)

    if invalidas:
        print(f"{contexto}: removendo UFs/linhas não oficiais: {invalidas}")

    return df[serie_uf.isin(UF_VALIDAS)].copy()


def calcular_percentual(parte: float, total: float) -> float:
    if pd.isna(total) or total == 0:
        return 0.0
    return float(parte) / float(total) * 100


def formatar_inteiro(valor: float) -> str:
    return f"{valor:,.0f}".replace(",", ".")


def formatar_moeda(valor: float) -> str:
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_percentual(valor: float) -> str:
    return f"{valor:.2f}%".replace(".", ",")


df_ufs


## 11. Carregar Políticas

Nesta etapa carregamos e tratamos cada política separadamente. A regra é: localizar o arquivo esperado, ler a aba correta, validar as colunas mínimas e deixar uma tabela pronta para uso nos indicadores do relatório.

### Carregar CAF

Origem: `caf_dap_ativos_ate_2026_03_gerado_em_20260410152650.xlsx`. Usamos a aba `TOTALIZADORES`, que já traz os valores consolidados por UF.

In [ ]:
arquivo_caf = arquivos_encontrados.get("caf_dap")
if arquivo_caf is None:
    raise FileNotFoundError("Arquivo CAF/DAP não encontrado.")

df_caf_totalizadores = pd.read_excel(
    arquivo_caf,
    sheet_name="TOTALIZADORES",
    engine="openpyxl",
    skiprows=6,
    nrows=27,
    na_values=["NA", "na", ""],
)
df_caf_totalizadores.columns = df_caf_totalizadores.columns.str.strip()

caf_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge_estados", "uf"]
caf_colunas_numericas = [
    "Soma de CAFs PF ATIVO",
    "Soma de CAFs PJ ATIVO",
    "Soma de QUANTIDADE DE MULHERES EM CAF ATIVO",
    "Soma de QUANTIDADE DE HOMENS EM CAF ATIVO",
]
validar_colunas(df_caf_totalizadores, caf_colunas_texto + caf_colunas_numericas, "CAF totalizadores")

for col in caf_colunas_texto:
    df_caf_totalizadores[col] = df_caf_totalizadores[col].astype(str).str.strip()
for col in caf_colunas_numericas:
    df_caf_totalizadores[col] = pd.to_numeric(df_caf_totalizadores[col], errors="coerce").fillna(0)

df_caf_totalizadores["uf"] = df_caf_totalizadores["uf"].str.upper()
df_caf_totalizadores = filtrar_ufs_validas(df_caf_totalizadores, "uf", "CAF totalizadores")
df_caf_totalizadores.head()


### Carregar PRONAF

Origem: `pronaf_gaia_20260414.xlsx`. Usamos a aba `Totalizadores`, com recortes por UF, ano e gênero.

In [ ]:
arquivo_pronaf = arquivos_encontrados.get("pronaf")
if arquivo_pronaf is None:
    raise FileNotFoundError("Arquivo PRONAF não encontrado.")

df_pronaf_totalizadores = pd.read_excel(
    arquivo_pronaf,
    sheet_name="Totalizadores",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_pronaf_totalizadores.columns = df_pronaf_totalizadores.columns.str.strip()

pronaf_colunas_texto = ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf"]
pronaf_colunas_numericas = [
    col
    for col in df_pronaf_totalizadores.columns
    if col not in pronaf_colunas_texto
]
validar_colunas(df_pronaf_totalizadores, pronaf_colunas_texto, "PRONAF totalizadores")

for col in pronaf_colunas_texto:
    df_pronaf_totalizadores[col] = df_pronaf_totalizadores[col].astype(str).str.strip()
for col in pronaf_colunas_numericas:
    df_pronaf_totalizadores[col] = pd.to_numeric(df_pronaf_totalizadores[col], errors="coerce").fillna(0)

df_pronaf_totalizadores["uf"] = df_pronaf_totalizadores["uf"].str.upper()
df_pronaf_totalizadores = filtrar_ufs_validas(df_pronaf_totalizadores, "uf", "PRONAF totalizadores")
df_pronaf_totalizadores.head()


### Carregar Mais Alimentos

Origem: `mais_alimentos_gaia_202604151554.xlsx`. Usamos a aba `Totalizadores`, que já consolida quantidade e valor por UF.

In [ ]:
arquivo_mais_alimentos = arquivos_encontrados.get("mais_alimentos")
if arquivo_mais_alimentos is None:
    raise FileNotFoundError("Arquivo Mais Alimentos não encontrado.")

df_mais_alimentos_totalizadores = pd.read_excel(
    arquivo_mais_alimentos,
    sheet_name="Totalizadores",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_mais_alimentos_totalizadores.columns = df_mais_alimentos_totalizadores.columns.str.strip()

mais_alimentos_colunas_texto = ["dt_referencia", "dt_geracao", "CD_ESTADO", "cod_ibge_uf"]
mais_alimentos_colunas_numericas = ["qtd_contratos", "valor_total_contratos"]
validar_colunas(
    df_mais_alimentos_totalizadores,
    mais_alimentos_colunas_texto + mais_alimentos_colunas_numericas,
    "Mais Alimentos totalizadores",
)

for col in mais_alimentos_colunas_texto:
    df_mais_alimentos_totalizadores[col] = df_mais_alimentos_totalizadores[col].astype(str).str.strip()
for col in mais_alimentos_colunas_numericas:
    df_mais_alimentos_totalizadores[col] = pd.to_numeric(df_mais_alimentos_totalizadores[col], errors="coerce").fillna(0)

df_mais_alimentos_totalizadores["uf"] = df_mais_alimentos_totalizadores["CD_ESTADO"].str.upper()
df_mais_alimentos_totalizadores = filtrar_ufs_validas(df_mais_alimentos_totalizadores, "uf", "Mais Alimentos totalizadores")
df_mais_alimentos_totalizadores.head()


### Carregar ATER

Origem: `ater_ate_2026_03_gerado_em_20260410151127.xlsx`. Usamos a aba `DADOS`, que é municipal, e criamos uma tabela agregada por UF.

In [ ]:
arquivo_ater = arquivos_encontrados.get("ater")
if arquivo_ater is None:
    raise FileNotFoundError("Arquivo ATER não encontrado.")

df_ater_dados = pd.read_excel(
    arquivo_ater,
    sheet_name="DADOS",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_ater_dados.columns = df_ater_dados.columns.str.strip()

ater_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf"]
ater_colunas_numericas = [
    "familias_com_ater_iniciada_no_ano",
    "familias_com_ater_recebida_no_ano",
]
validar_colunas(df_ater_dados, ater_colunas_texto + ater_colunas_numericas, "ATER dados")

for col in ater_colunas_texto:
    df_ater_dados[col] = df_ater_dados[col].astype(str).str.strip()
for col in ater_colunas_numericas:
    df_ater_dados[col] = pd.to_numeric(df_ater_dados[col], errors="coerce").fillna(0)

df_ater_dados["uf"] = df_ater_dados["uf"].str.upper()
df_ater_dados = filtrar_ufs_validas(df_ater_dados, "uf", "ATER dados")

df_ater_totalizadores = (
    df_ater_dados
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        familias_com_ater_iniciada_no_ano=("familias_com_ater_iniciada_no_ano", "sum"),
        familias_com_ater_recebida_no_ano=("familias_com_ater_recebida_no_ano", "sum"),
    )
)
df_ater_totalizadores.head()


### Carregar Garantia-Safra

Origem: `GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx`. Usamos a aba `DADOS`, que é municipal, e criamos uma tabela agregada por UF.

In [ ]:
arquivo_garantia_safra = arquivos_encontrados.get("garantia_safra")
if arquivo_garantia_safra is None:
    raise FileNotFoundError("Arquivo Garantia-Safra não encontrado.")

df_garantia_safra_dados = pd.read_excel(
    arquivo_garantia_safra,
    sheet_name="DADOS",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_garantia_safra_dados.columns = df_garantia_safra_dados.columns.str.strip()

garantia_safra_colunas_texto = [
    "dt_referencia",
    "dt_geracao",
    "uf",
    "cod_ibge_uf",
    "cod_ibge",
    "nome_municipio",
    "pagamento_liberado",
]
garantia_safra_colunas_numericas = [
    "Agricultores_aderidos",
    "Agricultores_com_pagamento liberado",
]
validar_colunas(
    df_garantia_safra_dados,
    garantia_safra_colunas_texto + garantia_safra_colunas_numericas,
    "Garantia-Safra dados",
)

for col in garantia_safra_colunas_texto:
    df_garantia_safra_dados[col] = df_garantia_safra_dados[col].astype(str).str.strip()
for col in garantia_safra_colunas_numericas:
    df_garantia_safra_dados[col] = pd.to_numeric(df_garantia_safra_dados[col], errors="coerce").fillna(0)

df_garantia_safra_dados["uf"] = df_garantia_safra_dados["uf"].str.upper()
df_garantia_safra_dados = filtrar_ufs_validas(df_garantia_safra_dados, "uf", "Garantia-Safra dados")

df_garantia_safra_totalizadores = (
    df_garantia_safra_dados
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        agricultores_aderidos=("Agricultores_aderidos", "sum"),
        agricultores_com_pagamento_liberado=("Agricultores_com_pagamento liberado", "sum"),
    )
)
df_garantia_safra_totalizadores.head()


### Carregar PNCF

Origem: `pncf_2026_03_gerado_em_20260410170006.xlsx`. Usamos a aba `DADOS`, que é municipal, e criamos uma tabela agregada por UF.

In [ ]:
import re
import unicodedata

arquivo_pncf = arquivos_encontrados.get("pncf")
if arquivo_pncf is None:
    raise FileNotFoundError("Arquivo PNCF não encontrado.")

def normalizar_nome_coluna(coluna: object) -> str:
    texto = unicodedata.normalize("NFKD", str(coluna))
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    texto = texto.lower().replace("º", "o").replace("°", "o")
    return re.sub(r"[^a-z0-9]+", "_", texto).strip("_")

df_pncf_dados = pd.read_excel(
    arquivo_pncf,
    sheet_name="DADOS",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_pncf_dados.columns = df_pncf_dados.columns.str.strip()

pncf_mapa_colunas = {
    "municipio": "municipio",
    "no_de_operacoes": "numero_operacoes",
    "valor_liberado_r": "valor_liberado",
    "valor_medio_liberado_r_operacao": "valor_medio_liberado",
}
df_pncf_dados = df_pncf_dados.rename(
    columns={
        coluna: pncf_mapa_colunas.get(normalizar_nome_coluna(coluna), coluna)
        for coluna in df_pncf_dados.columns
    }
)

pncf_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge", "sigla_uf", "nome_uf", "municipio"]
pncf_colunas_numericas = ["numero_operacoes", "valor_liberado", "valor_medio_liberado"]
validar_colunas(df_pncf_dados, pncf_colunas_texto + pncf_colunas_numericas, "PNCF dados")

for col in pncf_colunas_texto:
    df_pncf_dados[col] = df_pncf_dados[col].astype(str).str.strip()
for col in pncf_colunas_numericas:
    df_pncf_dados[col] = pd.to_numeric(df_pncf_dados[col], errors="coerce").fillna(0)

df_pncf_dados["sigla_uf"] = df_pncf_dados["sigla_uf"].str.upper()
df_pncf_dados = filtrar_ufs_validas(df_pncf_dados, "sigla_uf", "PNCF dados")

df_pncf_totalizadores = (
    df_pncf_dados
    .groupby("sigla_uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        numero_operacoes=("numero_operacoes", "sum"),
        valor_liberado=("valor_liberado", "sum"),
    )
    .rename(columns={"sigla_uf": "uf"})
)
df_pncf_totalizadores["valor_medio_liberado"] = df_pncf_totalizadores.apply(
    lambda row: row["valor_liberado"] / row["numero_operacoes"] if row["numero_operacoes"] else 0,
    axis=1,
)

df_pncf_totalizadores.head()


### Carregar PNRA

Origem: `PNRA_2026_2026_04_15.xlsx`. Usamos a aba `DADOS`, que é municipal, e criamos uma tabela agregada por UF.

In [ ]:
arquivo_pnra = arquivos_encontrados.get("pnra")
if arquivo_pnra is None:
    raise FileNotFoundError("Arquivo PNRA não encontrado.")

df_pnra_dados = pd.read_excel(
    arquivo_pnra,
    sheet_name="DADOS",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_pnra_dados.columns = df_pnra_dados.columns.str.strip()

pnra_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf"]
pnra_colunas_numericas = [
    "numero_familias_pa_criado_diferenciado_pnra",
    "numero_familias_pa_criado_tradicional_pnra",
    "numero_familias_reconhecimento_pnra",
    "numero_familias_regularizacao_pnra",
    "total_numero_familias_pnra",
]
validar_colunas(df_pnra_dados, pnra_colunas_texto + pnra_colunas_numericas, "PNRA dados")

for col in pnra_colunas_texto:
    df_pnra_dados[col] = df_pnra_dados[col].astype(str).str.strip()
for col in pnra_colunas_numericas:
    df_pnra_dados[col] = pd.to_numeric(df_pnra_dados[col], errors="coerce").fillna(0)

df_pnra_dados["uf"] = df_pnra_dados["uf"].str.upper()
df_pnra_dados = filtrar_ufs_validas(df_pnra_dados, "uf", "PNRA dados")

df_pnra_totalizadores = (
    df_pnra_dados
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        numero_familias_pa_criado_diferenciado_pnra=("numero_familias_pa_criado_diferenciado_pnra", "sum"),
        numero_familias_pa_criado_tradicional_pnra=("numero_familias_pa_criado_tradicional_pnra", "sum"),
        numero_familias_reconhecimento_pnra=("numero_familias_reconhecimento_pnra", "sum"),
        numero_familias_regularizacao_pnra=("numero_familias_regularizacao_pnra", "sum"),
        total_numero_familias_pnra=("total_numero_familias_pnra", "sum"),
    )
)
df_pnra_totalizadores.head()


### Resumo das políticas carregadas

In [ ]:
politicas_tratadas = {
    "CAF": df_caf_totalizadores,
    "PRONAF": df_pronaf_totalizadores,
    "Mais Alimentos": df_mais_alimentos_totalizadores,
    "ATER": df_ater_totalizadores,
    "Garantia-Safra": df_garantia_safra_totalizadores,
    "PNCF": df_pncf_totalizadores,
    "PNRA": df_pnra_totalizadores,
}

resumo_politicas = pd.DataFrame(
    [
        {
            "politica": nome,
            "linhas": len(df),
            "colunas": len(df.columns),
            "ufs": df["uf"].nunique() if "uf" in df.columns else None,
        }
        for nome, df in politicas_tratadas.items()
    ]
)
resumo_politicas

## 12. Calcular indicadores UF x Brasil

Aqui consolidamos os principais indicadores de todas as políticas carregadas. O resultado é uma tabela única, com valor da UF escolhida, valor Brasil e participação percentual da UF.

In [ ]:
UF_INTERESSE = normalizar_uf(UF_INTERESSE)
GERAR_TODAS_UFS = bool(GERAR_TODAS_UFS) or UF_INTERESSE in UF_ALIASES_LOTE

if GERAR_TODAS_UFS:
    print("Modo de geração: todas as 27 UFs do dicionário oficial.")
else:
    print(f"UF normalizada: {UF_INTERESSE} - {UF_NOMES[UF_INTERESSE]}")


## 13. Gerar relatório Word de teste

Esta geração `.docx` separa as políticas públicas em seções próprias, seguindo a lógica editorial do `documento_padrao_v1.docx` e os requisitos do PRD. O arquivo é salvo em `OUTPUT_DIR`, que no Colab aponta para o Google Drive e no modo local aponta para `relatorios_gerados/AAAAMM`.

In [ ]:
import re
import shutil
from pathlib import Path

INFORMACAO_INDISPONIVEL = "Informação não disponível nas bases atuais."

POLITICAS_RELATORIO = [
    {
        "nome": "CAF",
        "titulo": "CAF - Cadastro da Agricultura Familiar",
        "descricao": "Indicadores consolidados de CAF ativo, com recorte de pessoa física, pessoa jurídica e gênero.",
        "fonte": "CAF Transparência",
        "df_referencia": df_caf_totalizadores,
        "lacunas": [
            "Jovens em CAF ativo.",
            "Quilombolas, indígenas e povos e comunidades tradicionais com DAP.",
            "Principais produtos da agricultura familiar no estado.",
            "Produtos com maior valor total de produção no estado.",
        ],
    },
    {
        "nome": "PRONAF",
        "titulo": "PRONAF",
        "descricao": "Dados gerais de contratos, operações e volume financeiro do PRONAF.",
        "fonte": "GAIA",
        "df_referencia": df_pronaf_totalizadores,
        "lacunas": ["Série histórica anual detalhada no formato do documento padrão."],
    },
    {
        "nome": "Mais Alimentos",
        "titulo": "Mais Alimentos",
        "descricao": "Quantidade de contratos e valor total dos contratos do Mais Alimentos.",
        "fonte": "GAIA",
        "df_referencia": df_mais_alimentos_totalizadores,
        "lacunas": ["Série histórica anual no formato do documento padrão."],
    },
    {
        "nome": "ATER",
        "titulo": "ATER",
        "descricao": "Famílias com ATER iniciada e recebida no ano.",
        "fonte": "Base ATER/ANATER",
        "df_referencia": df_ater_totalizadores,
        "lacunas": [],
    },
    {
        "nome": "Garantia-Safra",
        "titulo": "Garantia-Safra",
        "descricao": "Agricultores aderidos e agricultores com pagamento liberado no Garantia-Safra.",
        "fonte": "Garantia-Safra",
        "df_referencia": df_garantia_safra_totalizadores,
        "lacunas": ["Seção sem equivalente direto no documento padrão v1; incluída porque há base disponível."],
    },
    {
        "nome": "PNCF",
        "titulo": "PNCF",
        "descricao": "Número de operações, valor liberado e valor médio liberado por operação.",
        "fonte": "PNCF",
        "df_referencia": df_pncf_totalizadores,
        "lacunas": ["Série histórica anual no formato do documento padrão."],
    },
    {
        "nome": "PNRA",
        "titulo": "PNRA - Reforma Agrária",
        "descricao": "Famílias por tipo de assentamento, reconhecimento e regularização no PNRA.",
        "fonte": "PNRA",
        "df_referencia": df_pnra_totalizadores,
        "lacunas": ["Quadros históricos anuais no formato do documento padrão."],
    },
]


def adicionar_cabecalho_tabela(tabela, cabecalhos: list[str]) -> None:
    for idx, titulo in enumerate(cabecalhos):
        cell = tabela.rows[0].cells[idx]
        cell.text = titulo
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.bold = True


def formatar_valor_relatorio(valor: float, formato: str) -> str:
    if formato == "moeda":
        return formatar_moeda(valor)
    return formatar_inteiro(valor)


def formatar_data_documento(valor) -> str:
    if valor is None or pd.isna(valor):
        return INFORMACAO_INDISPONIVEL

    texto = str(valor).strip()
    if not texto:
        return INFORMACAO_INDISPONIVEL

    data_ano_mes_dia = re.fullmatch(r"(\d{4})[_-](\d{2})[_-](\d{2})", texto)
    if data_ano_mes_dia:
        ano, mes, dia = data_ano_mes_dia.groups()
        return f"{dia}/{mes}/{ano}"

    data_ano_mes = re.fullmatch(r"(\d{4})[_-](\d{2})", texto)
    if data_ano_mes:
        ano, mes = data_ano_mes.groups()
        return f"{mes}/{ano}"

    data = pd.to_datetime(texto, errors="coerce")
    if not pd.isna(data):
        return data.strftime("%d/%m/%Y")
    return texto


def obter_primeiro_valor(df: pd.DataFrame, coluna: str):
    if coluna not in df.columns:
        return None
    serie = df[coluna].dropna()
    if serie.empty:
        return None
    return serie.iloc[0]


def adicionar_fonte_referencia(doc: Document, politica: dict) -> None:
    referencia = formatar_data_documento(obter_primeiro_valor(politica["df_referencia"], "dt_referencia"))
    geracao = formatar_data_documento(obter_primeiro_valor(politica["df_referencia"], "dt_geracao"))
    doc.add_paragraph(f"Fonte: {politica['fonte']}. Referência: {referencia}. Geração da base: {geracao}.")


def adicionar_lacunas(doc: Document, lacunas: list[str]) -> None:
    if not lacunas:
        return
    doc.add_paragraph("Lacunas em relação ao documento padrão:")
    for lacuna in lacunas:
        doc.add_paragraph(f"{lacuna} {INFORMACAO_INDISPONIVEL}", style="List Bullet")


def adicionar_tabela_indicadores(doc: Document, df_indicadores: pd.DataFrame, uf_label: str) -> None:
    table = doc.add_table(rows=1, cols=4)
    table.style = "Table Grid"
    adicionar_cabecalho_tabela(table, ["Indicador", "Brasil", uf_label, "Percentual em relação ao nacional"])

    for _, row in df_indicadores.iterrows():
        cells = table.add_row().cells
        cells[0].text = str(row["indicador"])
        cells[1].text = formatar_valor_relatorio(row["brasil"], row["formato"])
        cells[2].text = formatar_valor_relatorio(row["uf"], row["formato"])
        cells[3].text = formatar_percentual(row["percentual_uf_br"])


def filtrar_uf_tolerante(df: pd.DataFrame, politica: str, uf_codigo: str) -> pd.DataFrame:
    validar_colunas(df, ["uf"], politica)
    df_uf = df[df["uf"].astype(str).str.upper() == uf_codigo].copy()
    return df_uf


def somar_coluna(df: pd.DataFrame, coluna: str, contexto: str) -> float:
    validar_colunas(df, [coluna], contexto)
    if df.empty:
        return 0.0
    return pd.to_numeric(df[coluna], errors="coerce").fillna(0).sum()


def somar_colunas(df: pd.DataFrame, colunas: list[str], contexto: str) -> float:
    validar_colunas(df, colunas, contexto)
    return sum(somar_coluna(df, coluna, contexto) for coluna in colunas)


def dividir_com_seguranca(numerador: float, denominador: float) -> float:
    if pd.isna(denominador) or denominador == 0:
        return 0.0
    return float(numerador) / float(denominador)


def adicionar_indicador(linhas: list[dict], politica: str, indicador: str, valor_uf: float, valor_brasil: float, formato: str = "inteiro") -> None:
    linhas.append(
        {
            "politica": politica,
            "indicador": indicador,
            "uf": float(valor_uf),
            "brasil": float(valor_brasil),
            "percentual_uf_br": calcular_percentual(valor_uf, valor_brasil),
            "formato": formato,
        }
    )


def construir_indicadores_por_uf(uf_codigo: str) -> pd.DataFrame:
    uf_codigo = normalizar_uf(uf_codigo)
    linhas = []

    # CAF
    df_caf_uf = filtrar_uf_tolerante(df_caf_totalizadores, "CAF", uf_codigo)
    for indicador, coluna in [
        ("CAFs Pessoa Física ativos", "Soma de CAFs PF ATIVO"),
        ("CAFs Pessoa Jurídica ativos", "Soma de CAFs PJ ATIVO"),
        ("Mulheres em CAF ativo", "Soma de QUANTIDADE DE MULHERES EM CAF ATIVO"),
        ("Homens em CAF ativo", "Soma de QUANTIDADE DE HOMENS EM CAF ATIVO"),
    ]:
        adicionar_indicador(
            linhas,
            "CAF",
            indicador,
            somar_coluna(df_caf_uf, coluna, "CAF UF"),
            somar_coluna(df_caf_totalizadores, coluna, "CAF Brasil"),
        )

    # PRONAF
    pronaf_colunas_qtd_contratos = [
        "qtd_contratos_1_Feminino",
        "qtd_contratos_1_Masculino",
        "qtd_contratos_1_Sem_Identificacao",
    ]
    pronaf_colunas_valor_contratos = [
        "valor_total_contratos_1_Feminino",
        "valor_total_contratos_1_Masculino",
        "valor_total_contratos_1_Sem_Identificacao",
    ]
    pronaf_colunas_qtd_operacoes = [
        "qtd_operacoes_1_Feminino",
        "qtd_operacoes_1_Masculino",
        "qtd_operacoes_1_Sem_Identificacao",
    ]
    df_pronaf_uf = filtrar_uf_tolerante(df_pronaf_totalizadores, "PRONAF", uf_codigo)
    adicionar_indicador(
        linhas,
        "PRONAF",
        "Contratos",
        somar_colunas(df_pronaf_uf, pronaf_colunas_qtd_contratos, "PRONAF UF"),
        somar_colunas(df_pronaf_totalizadores, pronaf_colunas_qtd_contratos, "PRONAF Brasil"),
    )
    adicionar_indicador(
        linhas,
        "PRONAF",
        "Valor total dos contratos",
        somar_colunas(df_pronaf_uf, pronaf_colunas_valor_contratos, "PRONAF UF"),
        somar_colunas(df_pronaf_totalizadores, pronaf_colunas_valor_contratos, "PRONAF Brasil"),
        formato="moeda",
    )
    adicionar_indicador(
        linhas,
        "PRONAF",
        "Operações",
        somar_colunas(df_pronaf_uf, pronaf_colunas_qtd_operacoes, "PRONAF UF"),
        somar_colunas(df_pronaf_totalizadores, pronaf_colunas_qtd_operacoes, "PRONAF Brasil"),
    )

    # Mais Alimentos
    df_mais_uf = filtrar_uf_tolerante(df_mais_alimentos_totalizadores, "Mais Alimentos", uf_codigo)
    for indicador, coluna, formato in [
        ("Contratos", "qtd_contratos", "inteiro"),
        ("Valor total dos contratos", "valor_total_contratos", "moeda"),
    ]:
        adicionar_indicador(
            linhas,
            "Mais Alimentos",
            indicador,
            somar_coluna(df_mais_uf, coluna, "Mais Alimentos UF"),
            somar_coluna(df_mais_alimentos_totalizadores, coluna, "Mais Alimentos Brasil"),
            formato=formato,
        )

    # ATER
    df_ater_uf = filtrar_uf_tolerante(df_ater_totalizadores, "ATER", uf_codigo)
    for indicador, coluna in [
        ("Famílias com ATER iniciada no ano", "familias_com_ater_iniciada_no_ano"),
        ("Famílias com ATER recebida no ano", "familias_com_ater_recebida_no_ano"),
    ]:
        adicionar_indicador(
            linhas,
            "ATER",
            indicador,
            somar_coluna(df_ater_uf, coluna, "ATER UF"),
            somar_coluna(df_ater_totalizadores, coluna, "ATER Brasil"),
        )

    # Garantia-Safra
    df_gs_uf = filtrar_uf_tolerante(df_garantia_safra_totalizadores, "Garantia-Safra", uf_codigo)
    for indicador, coluna in [
        ("Agricultores aderidos", "agricultores_aderidos"),
        ("Agricultores com pagamento liberado", "agricultores_com_pagamento_liberado"),
    ]:
        adicionar_indicador(
            linhas,
            "Garantia-Safra",
            indicador,
            somar_coluna(df_gs_uf, coluna, "Garantia-Safra UF"),
            somar_coluna(df_garantia_safra_totalizadores, coluna, "Garantia-Safra Brasil"),
        )

    # PNCF
    df_pncf_uf = filtrar_uf_tolerante(df_pncf_totalizadores, "PNCF", uf_codigo)
    pncf_ops_uf = somar_coluna(df_pncf_uf, "numero_operacoes", "PNCF UF")
    pncf_ops_br = somar_coluna(df_pncf_totalizadores, "numero_operacoes", "PNCF Brasil")
    pncf_val_uf = somar_coluna(df_pncf_uf, "valor_liberado", "PNCF UF")
    pncf_val_br = somar_coluna(df_pncf_totalizadores, "valor_liberado", "PNCF Brasil")
    adicionar_indicador(linhas, "PNCF", "Operações", pncf_ops_uf, pncf_ops_br)
    adicionar_indicador(linhas, "PNCF", "Valor liberado", pncf_val_uf, pncf_val_br, formato="moeda")
    adicionar_indicador(
        linhas,
        "PNCF",
        "Valor médio liberado por operação",
        dividir_com_seguranca(pncf_val_uf, pncf_ops_uf),
        dividir_com_seguranca(pncf_val_br, pncf_ops_br),
        formato="moeda",
    )

    # PNRA
    df_pnra_uf = filtrar_uf_tolerante(df_pnra_totalizadores, "PNRA", uf_codigo)
    for indicador, coluna in [
        ("Famílias em PA criado diferenciado", "numero_familias_pa_criado_diferenciado_pnra"),
        ("Famílias em PA criado tradicional", "numero_familias_pa_criado_tradicional_pnra"),
        ("Famílias em reconhecimento", "numero_familias_reconhecimento_pnra"),
        ("Famílias em regularização", "numero_familias_regularizacao_pnra"),
        ("Total de famílias", "total_numero_familias_pnra"),
    ]:
        adicionar_indicador(
            linhas,
            "PNRA",
            indicador,
            somar_coluna(df_pnra_uf, coluna, "PNRA UF"),
            somar_coluna(df_pnra_totalizadores, coluna, "PNRA Brasil"),
        )

    return pd.DataFrame(linhas)


def politica_tem_registro_uf(politica: dict, uf_codigo: str) -> bool:
    df_ref = politica["df_referencia"]
    if "uf" not in df_ref.columns:
        return False
    return uf_codigo in set(df_ref["uf"].astype(str).str.upper())


RANKING_INDICADOR_PRINCIPAL = {
    "CAF": "CAFs Pessoa Física ativos",
    "PRONAF": "Valor total dos contratos",
    "Mais Alimentos": "Valor total dos contratos",
    "ATER": "Famílias com ATER recebida no ano",
    "Garantia-Safra": "Agricultores com pagamento liberado",
    "PNCF": "Valor liberado",
    "PNRA": "Total de famílias",
}


def obter_ufs_para_ranking() -> list[str]:
    return df_ufs["sigla"].tolist()


def construir_ranking_politicas() -> dict[str, pd.DataFrame]:
    registros = []
    for uf in obter_ufs_para_ranking():
        df_uf = construir_indicadores_por_uf(uf)
        if df_uf.empty:
            continue

        for politica, indicador in RANKING_INDICADOR_PRINCIPAL.items():
            linha = df_uf[(df_uf["politica"] == politica) & (df_uf["indicador"] == indicador)]
            if linha.empty:
                continue

            row = linha.iloc[0]
            registros.append(
                {
                    "politica": politica,
                    "indicador_ranking": indicador,
                    "uf": uf,
                    "valor_uf": float(row["uf"]),
                    "percentual_uf_br": float(row["percentual_uf_br"]),
                }
            )

    if not registros:
        return {}

    df_rank = pd.DataFrame(registros)
    df_rank["posicao"] = (
        df_rank.groupby("politica")["valor_uf"]
        .rank(method="min", ascending=False)
        .astype(int)
    )
    df_rank["total_ufs"] = 27

    return {
        politica: grupo.sort_values(["posicao", "uf"]).reset_index(drop=True)
        for politica, grupo in df_rank.groupby("politica")
    }


def adicionar_avaliacao_ranking(
    doc: Document,
    ranking_politicas: dict[str, pd.DataFrame],
    politica_nome: str,
    uf_codigo: str,
    tem_registro_uf: bool,
) -> None:
    ranking = ranking_politicas.get(politica_nome)
    if ranking is None or ranking.empty:
        return

    indicador = RANKING_INDICADOR_PRINCIPAL.get(politica_nome)
    if not tem_registro_uf:
        doc.add_paragraph(
            "Avaliação da posição no ranking: "
            f"{uf_codigo} não possui registro nesta base para o indicador de ranking "
            f"({indicador}). Por isso, a UF não recebe uma colocação interpretável; "
            "o universo de comparação considera as 27 UFs oficiais."
        )
        return

    linha = ranking[ranking["uf"] == uf_codigo]
    if linha.empty:
        return

    posicao = int(linha.iloc[0]["posicao"])
    total_ufs = int(linha.iloc[0]["total_ufs"])
    percentual = float(linha.iloc[0]["percentual_uf_br"])

    doc.add_paragraph(
        "Avaliação da posição no ranking: "
        f"{uf_codigo} está na {posicao}ª posição entre {total_ufs} UFs, "
        f"com participação de {formatar_percentual(percentual)} no total nacional "
        f"para o indicador de ranking ({indicador})."
    )


def adicionar_sumario_executivo(doc: Document, nome_uf: str, df_indicadores: pd.DataFrame) -> None:
    caf_pf = df_indicadores[(df_indicadores["politica"] == "CAF") & (df_indicadores["indicador"] == "CAFs Pessoa Física ativos")]
    pronaf_valor = df_indicadores[(df_indicadores["politica"] == "PRONAF") & (df_indicadores["indicador"] == "Valor total dos contratos")]

    frases = []
    if not caf_pf.empty:
        row = caf_pf.iloc[0]
        frases.append(
            f"{nome_uf} possui {formatar_inteiro(row['uf'])} CAFs Pessoa Física ativos, "
            f"equivalentes a {formatar_percentual(row['percentual_uf_br'])} do total nacional."
        )
    if not pronaf_valor.empty:
        row = pronaf_valor.iloc[0]
        frases.append(
            f"No PRONAF, o volume financeiro consolidado para a UF é {formatar_moeda(row['uf'])}, "
            f"equivalente a {formatar_percentual(row['percentual_uf_br'])} do Brasil."
        )

    if frases:
        doc.add_heading("Sumário Executivo", level=1)
        doc.add_paragraph(" ".join(frases))


def gerar_relatorio_uf(uf_codigo: str, ranking_politicas: dict[str, pd.DataFrame] | None = None):
    uf_codigo = normalizar_uf(uf_codigo)
    nome_uf = UF_NOMES[uf_codigo]
    df_indicadores = construir_indicadores_por_uf(uf_codigo)
    ranking_politicas = ranking_politicas or {}

    doc = Document()

    doc.add_paragraph("Versão 1")
    titulo = doc.add_heading("Relatório Estadual de Monitoramento", level=0)
    titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER
    subtitulo = doc.add_paragraph(nome_uf)
    subtitulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph(f"Gerado em: {RUN_TIMESTAMP.strftime('%d/%m/%Y %H:%M')}")
    doc.add_paragraph(
        "Documento preliminar gerado a partir das bases de políticas públicas disponíveis. "
        "A estrutura segue o documento padrão v1 como referência editorial."
    )

    adicionar_sumario_executivo(doc, nome_uf, df_indicadores)

    for numero, politica in enumerate(POLITICAS_RELATORIO, start=1):
        df_secao = df_indicadores[df_indicadores["politica"] == politica["nome"]].copy()

        doc.add_heading(f"{numero}. {politica['titulo']}", level=1)
        doc.add_paragraph(politica["descricao"])

        tem_registro_uf = politica_tem_registro_uf(politica, uf_codigo)
        if not tem_registro_uf:
            doc.add_paragraph(
                "Sem registros para a UF nesta base. Os indicadores estaduais aparecem como zero "
                "e devem ser interpretados como ausência de registro na base atual."
            )

        if df_secao.empty:
            doc.add_paragraph(INFORMACAO_INDISPONIVEL)
        else:
            adicionar_tabela_indicadores(doc, df_secao, nome_uf)

        if INCLUIR_RANKING:
            adicionar_avaliacao_ranking(doc, ranking_politicas, politica["nome"], uf_codigo, tem_registro_uf)

        adicionar_fonte_referencia(doc, politica)
        adicionar_lacunas(doc, politica["lacunas"])

    nome_relatorio = f"relatorio_estadual_monitoramento_{uf_codigo.lower()}_{RUN_DATETIME}.docx"
    caminho_relatorio = OUTPUT_DIR / nome_relatorio
    doc.save(caminho_relatorio)
    return caminho_relatorio


ufs_para_gerar = df_ufs["sigla"].tolist() if GERAR_TODAS_UFS else [UF_INTERESSE]
ranking_politicas = construir_ranking_politicas() if INCLUIR_RANKING else {}

relatorios = []
for uf_codigo in ufs_para_gerar:
    caminho = gerar_relatorio_uf(uf_codigo, ranking_politicas)
    relatorios.append(caminho)
    print(f"Relatório Word gerado: {caminho}")

if GERAR_TODAS_UFS and esta_no_colab():
    try:
        from google.colab import files

        zip_base = OUTPUT_DIR / f"relatorios_estaduais_{RUN_DATETIME}"
        zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(OUTPUT_DIR)))
        print(f"ZIP gerado: {zip_path}")
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Não foi possível disponibilizar download automático: {exc}")
